<a href="https://colab.research.google.com/github/alexandrumoldovan1/housing-prices-ml/blob/main/notebooks/01_data_loading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Install required libraries
!pip install ydata-profiling -q
!pip install pyarrow -q
print("Libraries installed successfully!")

Libraries installed successfully!


In [8]:
# Import standard libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define project paths
DRIVE_PATH = '/content/drive/MyDrive/ColabProjects/housing-prices-ml'
PROCESSED_DATA_PATH = f'{DRIVE_PATH}/processed_data'
MODELS_PATH = f'{DRIVE_PATH}/models'
OUTPUTS_PATH = f'{DRIVE_PATH}/outputs'

print("Libraries imported and Drive mounted!")
print(f"Project path: {DRIVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries imported and Drive mounted!
Project path: /content/drive/MyDrive/ColabProjects/housing-prices-ml


In [9]:
# Clone the GitHub repository to access the data files
!git clone https://github.com/alexandrumoldovan1/housing-prices-ml.git

# Define the data path inside the cloned repo
REPO_PATH = '/content/housing-prices-ml'
DATA_PATH = f'{REPO_PATH}/data'

print(" Repository cloned successfully!")
print(f" Data path: {DATA_PATH}")

fatal: destination path 'housing-prices-ml' already exists and is not an empty directory.
 Repository cloned successfully!
 Data path: /content/housing-prices-ml/data


In [10]:
# Explore the data folder structure
print(" Data folder structure:\n")

for year_folder in sorted(os.listdir(DATA_PATH)):
    year_path = os.path.join(DATA_PATH, year_folder)
    if os.path.isdir(year_path):
        files = sorted(os.listdir(year_path))
        print(f" {year_folder}/  ({len(files)} files)")
        for file in files:
            file_path = os.path.join(year_path, file)
            size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f"    {file}  ({size_mb:.2f} MB)")
        print()

 Data folder structure:

 2023/  (5 files)
    2023_bronx.xlsx  (0.60 MB)
    2023_brooklyn.xlsx  (2.13 MB)
    2023_manhattan.xlsx  (1.48 MB)
    2023_queens.xlsx  (2.37 MB)
    2023_staten_island.xlsx  (0.78 MB)

 2024/  (5 files)
    2024_bronx.xlsx  (0.68 MB)
    2024_brooklyn.xlsx  (2.35 MB)
    2024_manhattan.xlsx  (1.76 MB)
    2024_queens.xlsx  (2.64 MB)
    2024_staten_island.xlsx  (0.84 MB)

 2025/  (5 files)
    2025_bronx.xlsx  (0.68 MB)
    2025_brooklyn.xlsx  (2.31 MB)
    2025_manhattan.xlsx  (1.70 MB)
    2025_queens.xlsx  (2.66 MB)
    2025_staten_island.xlsx  (0.81 MB)



In [16]:
# Load all 15 Excel files and concatenate into a single DataFrame
print("Loading Excel files...\n")

all_dataframes = []
years = ['2023', '2024', '2025']
boroughs = ['bronx', 'brooklyn', 'manhattan', 'queens', 'staten_island']

for year in years:
    for borough in boroughs:
        file_path = f'{DATA_PATH}/{year}/{year}_{borough}.xlsx'

        try:
            # NYC sales files have metadata rows; actual headers are on row 6 (index 5)
            df = pd.read_excel(file_path, skiprows=6, header=0)

            # Drop fully empty rows
            df = df.dropna(how='all').reset_index(drop=True)

            # Add identifier columns
            df['sale_year'] = int(year)
            df['borough_name'] = borough

            all_dataframes.append(df)
            print(f"Loaded {year}/{borough}: {df.shape[0]:,} rows x {df.shape[1]} columns")

        except Exception as e:
            print(f"Error loading {year}/{borough}: {e}")

# Concatenate all DataFrames
df_master = pd.concat(all_dataframes, ignore_index=True)

# Clean column names (strip whitespace, replace newlines)
df_master.columns = df_master.columns.str.strip().str.replace('\n', ' ', regex=False).str.replace('\r', '', regex=False)

print(f"\nMASTER DATAFRAME:")
print(f"   Total rows: {df_master.shape[0]:,}")
print(f"   Total columns: {df_master.shape[1]}")

Loading Excel files...

Loaded 2023/bronx: 5,920 rows x 23 columns
Loaded 2023/brooklyn: 21,635 rows x 23 columns
Loaded 2023/manhattan: 17,009 rows x 23 columns
Loaded 2023/queens: 24,256 rows x 23 columns
Loaded 2023/staten_island: 7,591 rows x 23 columns
Loaded 2024/bronx: 6,203 rows x 23 columns
Loaded 2024/brooklyn: 21,994 rows x 23 columns
Loaded 2024/manhattan: 17,379 rows x 23 columns
Loaded 2024/queens: 25,003 rows x 23 columns
Loaded 2024/staten_island: 7,664 rows x 23 columns
Loaded 2025/bronx: 6,818 rows x 23 columns
Loaded 2025/brooklyn: 23,472 rows x 23 columns
Loaded 2025/manhattan: 19,724 rows x 23 columns
Loaded 2025/queens: 27,258 rows x 23 columns
Loaded 2025/staten_island: 7,796 rows x 23 columns

MASTER DATAFRAME:
   Total rows: 239,722
   Total columns: 23


In [18]:
# Display basic info about the master DataFrame
print("COLUMN NAMES AND DATA TYPES:\n")
print(df_master.dtypes)

print("\n" + "="*60)
print("FIRST 5 ROWS:")
print("="*60)
df_master.head()

COLUMN NAMES AND DATA TYPES:

BOROUGH                                  float64
NEIGHBORHOOD                              object
BUILDING CLASS CATEGORY                   object
TAX CLASS AT PRESENT                      object
BLOCK                                    float64
LOT                                      float64
EASE-MENT                                float64
BUILDING CLASS AT PRESENT                 object
ADDRESS                                   object
APARTMENT NUMBER                          object
ZIP CODE                                 float64
RESIDENTIAL UNITS                        float64
COMMERCIAL UNITS                         float64
TOTAL  UNITS                             float64
LAND  SQUARE FEET                        float64
GROSS  SQUARE FEET                       float64
YEAR BUILT                               float64
TAX CLASS AT TIME OF SALE                float64
BUILDING CLASS AT TIME OF SALE            object
SALE PRICE                             

,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASE-MENT,BUILDING CLASS AT PRESENT,ADDRESS,APARTMENT NUMBER,...,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE,sale_year,borough_name
0,2.0,BATHGATE,01 ONE FAMILY DWELLINGS,1,3030.0,66.0,NaN,A1,4453 PARK AVENUE,NaN,...,1.0,1646.0,1497.0,1899.0,1.0,A1,215000.0,2023-04-18,2023,bronx
1,2.0,BATHGATE,01 ONE FAMILY DWELLINGS,1,3030.0,66.0,NaN,A1,4453 PARK AVENUE,NaN,...,1.0,1646.0,1497.0,1899.0,1.0,A1,570000.0,2023-08-23,2023,bronx
2,2.0,BATHGATE,01 ONE FAMILY DWELLINGS,1,3035.0,52.0,NaN,A1,461 EAST 178 STREET,NaN,...,1.0,1782.0,1548.0,1899.0,1.0,A1,0.0,2023-04-14,2023,bronx
3,2.0,BATHGATE,01 ONE FAMILY DWELLINGS,1,3053.0,86.0,NaN,S0,2364 WASHINGTON AVENUE,NaN,...,3.0,1911.0,4080.0,1931.0,1.0,S0,0.0,2023-10-24,2023,bronx
4,2.0,BATHGATE,02 TWO FAMILY DWELLINGS,1,2904.0,22.0,NaN,B9,454 EAST 172 STREET,NaN,...,2.0,1658.0,1428.0,1901.0,1.0,B9,350000.0,2023-06-26,2023,bronx


In [19]:
# Save the master DataFrame as Parquet for use in other notebooks
output_file = f'{PROCESSED_DATA_PATH}/raw_data.parquet'

df_master.to_parquet(output_file, compression='snappy', index=False)

# Verify the file was saved
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"File saved successfully!")
print(f"   Path: {output_file}")
print(f"   Size: {file_size_mb:.2f} MB")
print(f"   Rows: {df_master.shape[0]:,}")
print(f"   Columns: {df_master.shape[1]}")

File saved successfully!
   Path: /content/drive/MyDrive/ColabProjects/housing-prices-ml/processed_data/raw_data.parquet
   Size: 4.79 MB
   Rows: 239,722
   Columns: 23


In [20]:
# Final summary of Notebook 01
print("="*60)
print("NOTEBOOK 01 - DATA LOADING SUMMARY")
print("="*60)

print(f"\nTotal records loaded: {df_master.shape[0]:,}")
print(f"Total columns: {df_master.shape[1]}")
print(f"Years covered: {sorted(df_master['sale_year'].unique())}")
print(f"Boroughs covered: {sorted(df_master['borough_name'].unique())}")

print(f"\nRecords per year:")
print(df_master['sale_year'].value_counts().sort_index().to_string())

print(f"\nRecords per borough:")
print(df_master['borough_name'].value_counts().to_string())

print(f"\nOutput file saved at:")
print(f"   {output_file}")

print("\n" + "="*60)
print("Notebook 01 completed successfully.")
print("Next step: Notebook 02 - Exploratory Data Analysis (EDA)")
print("="*60)

NOTEBOOK 01 - DATA LOADING SUMMARY

Total records loaded: 239,722
Total columns: 23
Years covered: [np.int64(2023), np.int64(2024), np.int64(2025)]
Boroughs covered: ['bronx', 'brooklyn', 'manhattan', 'queens', 'staten_island']

Records per year:
sale_year
2023    76411
2024    78243
2025    85068

Records per borough:
borough_name
queens           76517
brooklyn         67101
manhattan        54112
staten_island    23051
bronx            18941

Output file saved at:
   /content/drive/MyDrive/ColabProjects/housing-prices-ml/processed_data/raw_data.parquet

Notebook 01 completed successfully.
Next step: Notebook 02 - Exploratory Data Analysis (EDA)
